# Week 2 Exercise — AI Real Estate Investment Advisor

This notebook implements an **AI Real Estate Investment Advisor** that helps users analyze property investments.

Features:

- Gradio chat interface
- Streaming LLM responses
- Model switching via OpenRouter
- System prompt personas
- Investment calculation tools:
  - Rental Yield
  - Mortgage Estimator
  - Investment Score

The assistant helps evaluate real estate opportunities and financial returns.

In [ ]:
import os
import time
import re
import gradio as gr

from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found")

print("OpenRouter key loaded successfully")

In [ ]:
def get_client():

    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY
    )

In [ ]:
MODELS = {

    "GPT-4o Mini": "openai/gpt-4o-mini",

    "Llama 3.1": "meta-llama/llama-3.1-8b-instruct",

    "Mixtral": "mistralai/mixtral-8x7b-instruct"

}

In [ ]:
def calculate_rental_yield(price, monthly_rent):

    annual_rent = monthly_rent * 12

    yield_percent = (annual_rent / price) * 100

    return f"""
Rental Yield Analysis

Property Price: ${price}
Monthly Rent: ${monthly_rent}

Annual Rent: ${annual_rent}

Rental Yield: {yield_percent:.2f}%
"""

In [ ]:
def calculate_mortgage(price, down_payment, rate, years):

    loan = price - down_payment

    monthly_rate = rate / 100 / 12

    months = years * 12

    payment = loan * (monthly_rate * (1 + monthly_rate)**months) / ((1 + monthly_rate)**months - 1)

    return f"""
Mortgage Estimate

Property Price: ${price}
Down Payment: ${down_payment}

Loan Amount: ${loan}

Interest Rate: {rate}%
Loan Term: {years} years

Estimated Monthly Payment: ${payment:.2f}
"""

In [ ]:
def investment_score(price, rent, growth):

    annual_rent = rent * 12

    yield_percent = (annual_rent / price) * 100

    score = 0

    if yield_percent > 8:
        score += 4
    elif yield_percent > 5:
        score += 3
    elif yield_percent > 3:
        score += 2
    else:
        score += 1

    if growth > 5:
        score += 3
    elif growth > 3:
        score += 2
    else:
        score += 1

    if price < 500000:
        score += 2
    else:
        score += 1

    return f"""
Investment Evaluation

Property Price: ${price}
Monthly Rent: ${rent}
Expected Growth: {growth}%

Rental Yield: {yield_percent:.2f}%

Investment Score: {score}/10
"""

In [ ]:
def maybe_use_tool(message):

    yield_pattern = re.compile(
        r"yield\s+for\s+\$?(\d+)[^\d]+\$?(\d+)",
        re.IGNORECASE
    )

    mortgage_pattern = re.compile(
        r"mortgage.*?(\d+).*?(\d+).*?(\d+).*?(\d+)",
        re.IGNORECASE
    )

    invest_pattern = re.compile(
        r"evaluate.*?(\d+).*?(\d+).*?(\d+)",
        re.IGNORECASE
    )

    yield_match = yield_pattern.search(message)

    if yield_match:

        price = int(yield_match.group(1))
        rent = int(yield_match.group(2))

        result = calculate_rental_yield(price, rent)

        return f"{message}\n\n[Tool Used: Rental Yield]\n{result}"


    mortgage_match = mortgage_pattern.search(message)

    if mortgage_match:

        price = int(mortgage_match.group(1))
        down = int(mortgage_match.group(2))
        rate = float(mortgage_match.group(3))
        years = int(mortgage_match.group(4))

        result = calculate_mortgage(price, down, rate, years)

        return f"{message}\n\n[Tool Used: Mortgage Calculator]\n{result}"


    invest_match = invest_pattern.search(message)

    if invest_match:

        price = int(invest_match.group(1))
        rent = int(invest_match.group(2))
        growth = float(invest_match.group(3))

        result = investment_score(price, rent, growth)

        return f"{message}\n\n[Tool Used: Investment Analyzer]\n{result}"

    return message

In [ ]:
SYSTEM_PROMPTS = {

"General Advisor":
"You are a helpful AI real estate advisor helping users analyze property investments.",

"Investment Analyst":
"You analyze properties focusing on ROI, rental yield, and financial risks.",

"Beginner Guide":
"You explain real estate investing concepts in a very simple beginner-friendly way.",

"Market Expert":
"You specialize in real estate markets, long-term appreciation, and investment strategy."

}

In [ ]:
def build_messages(history, user_msg, persona):

    messages = [

        {"role": "system", "content": SYSTEM_PROMPTS[persona]}

    ]

    for user, assistant in history:

        messages.append({"role": "user", "content": user})

        messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": user_msg})

    return messages

In [ ]:
def stream_response(model_choice, messages):

    client = get_client()

    model = MODELS[model_choice]

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    reply = ""

    for chunk in stream:

        # Skip empty chunks
        if not chunk.choices:
            continue

        delta = chunk.choices[0].delta

        if delta and getattr(delta, "content", None):

            reply += delta.content

            yield reply

        time.sleep(0.01)

In [ ]:
def chat_fn(user_msg, history, model_choice, persona):

    if not user_msg:
        return history

    user_msg = maybe_use_tool(user_msg)

    messages = build_messages(history, user_msg, persona)

    partial = ""

    for chunk in stream_response(model_choice, messages):

        partial = chunk

        yield history + [(user_msg, partial)]

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
# 🏠 AI Real Estate Investment Advisor

Ask questions about property investments, returns, or mortgages.

### Example prompts

• calculate yield for 300000 with 2000 rent  
• mortgage for 400000 with 80000 down 6 30  
• evaluate investment 350000 rent 2200 growth 5  

Features:
- OpenRouter model switching
- Real estate advisor personas
- Investment tools
- Streaming AI responses
""")

    with gr.Row():

        model_choice = gr.Dropdown(

            list(MODELS.keys()),

            value="GPT-4o Mini",

            label="Model"
        )

        persona = gr.Dropdown(

            list(SYSTEM_PROMPTS.keys()),

            value="General Advisor",

            label="Advisor Type"
        )

    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(

        placeholder="Ask about real estate investment...",

        label="Your message"
    )

    send = gr.Button("Send", variant="primary")

    clear = gr.Button("Clear")

    state = gr.State([])

    msg.submit(

        chat_fn,

        [msg, state, model_choice, persona],

        chatbot

    ).then(

        lambda chat: chat,

        chatbot,

        state

    ).then(

        lambda: "",

        None,

        msg
    )

    send.click(

        chat_fn,

        [msg, state, model_choice, persona],

        chatbot

    ).then(

        lambda chat: chat,

        chatbot,

        state

    ).then(

        lambda: "",

        None,

        msg
    )

    clear.click(

        lambda: ([], []),

        None,

        [chatbot, state],

        queue=False
    )

In [ ]:
demo.launch()